In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Carica il dataset
dataset_path = "Food_Delivery_Times.csv"
df = pd.read_csv(dataset_path)

# Rimuove righe con valori nulli
df.dropna(inplace=True)

# Elimina la colonna 'Order_ID'
df.drop(columns=['Order_ID'], inplace=True)

# Separazione tra feature e target
X = df.drop(columns=['Delivery_Time_min'])
y = df['Delivery_Time_min']

# Identificare colonne numeriche e categoriche
numerical_features = ['Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs']
categorical_features = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

# Preprocessing: standardizzazione numeriche e one-hot encoding categoriche
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Creazione pipeline con Ridge Regression
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

# Suddivisione in training e test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definizione della griglia di iperparametri
param_grid = {
    'model__alpha': [0.01, 0.1, 1, 10, 100]
}

# Ottimizzazione con GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

# Predizioni sul test set
y_pred = grid_search.best_estimator_.predict(X_test)

# Calcolo metriche
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Stampa risultati
print(f"Miglior alpha: {grid_search.best_params_}")
print(f"Mean Absolute Error (MAE): {mae}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"R-squared (R2): {r2}")

# Recuperare il modello addestrato
model = grid_search.best_estimator_.named_steps["model"]

# Ottenere i nomi delle feature trasformate
feature_names = numerical_features + list(grid_search.best_estimator_.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(categorical_features))

# Estrarre i coefficienti della regressione lineare
coefficients = model.coef_

# Creare un DataFrame per visualizzare meglio i pesi
coef_df = pd.DataFrame({"Feature": feature_names, "Peso": coefficients})
coef_df = coef_df.sort_values(by="Peso", ascending=False)  # Ordinare per importanza

# Stampare i pesi
print(coef_df)


Miglior alpha: {'model__alpha': 10}
Mean Absolute Error (MAE): 5.4337572772278815
Mean Squared Error (MSE): 68.14279791002562
Root Mean Squared Error (RMSE): 8.254865105501459
R-squared (R2): 0.8332776212000605
                   Feature       Peso
0              Distance_km  16.654015
1     Preparation_Time_min   6.645406
8       Traffic_Level_High   6.113240
6            Weather_Snowy   3.790848
4            Weather_Foggy   3.210471
12     Time_of_Day_Evening   1.100691
16        Vehicle_Type_Car   0.793340
11   Time_of_Day_Afternoon   0.083363
15       Vehicle_Type_Bike  -0.039630
13     Time_of_Day_Morning  -0.130532
5            Weather_Rainy  -0.132494
10    Traffic_Level_Medium  -0.386203
17    Vehicle_Type_Scooter  -0.753710
14       Time_of_Day_Night  -1.053522
2   Courier_Experience_yrs  -1.764564
7            Weather_Windy  -2.054882
3            Weather_Clear  -4.813944
9        Traffic_Level_Low  -5.727037
